# TeAAL specifications on variants of Loops' SpMM implementations (Work Oriented)

Description: All threads are assigned (`TOTAL_WORK` / `Number of Processors`) work items. Each thread then sequentially processes assigned work items in a loop. `TOTAL_WORK` = `NNZ` + `NUM_ROWS`.

Template: DNE in Loops

Scheduler: https://github.com/gunrock/loops/blob/main/include/loops/schedule/work_oriented.hxx

## Imports

Import the necessary modules.

In [ ]:
# HiFiber boilerplate

from fibertree_bootstrap import *

fibertree_bootstrap(style="tree", animation='movie')

# Compilation boilerplate

import os
import sys
sys.path.insert(0, "../..")

from src import utils

## Initialization

Initialize the input tensors.

For simplicity, the size of a thread warp is the same as the size of a thread block (`WARP_SIZE = BLOCK_SIZE`). Suppose that each GPU SM processes 1 thread warp/block per cycle.

In [ ]:
K = 4
M = 4
N = 4

# GPU Kernel Configuration
BLOCK_SIZE = 2 # Number of threads per block
GRID_SIZE = 2 # Number of thread blocks

print(f"GPU Kernel Configuration\n \
        BLOCK_SIZE (Number of threads per block): {BLOCK_SIZE} \n \
        GRID_SIZE (Number of thread blocks): {GRID_SIZE}")

seed = 1

#A_MK = Tensor.fromRandom(rank_ids=["M", "K"], shape=[M, K], seed=seed, density=[0.9, 0.6], name="A")
A_MK = Tensor.fromUncompressed(rank_ids=["M", "K"], root=[[0,0,8,0],[4,1,0,10],[8,4,2,0],[9,7,5,0]], shape=[M, K], name="A")
B_KN = Tensor.fromRandom(rank_ids=["K", "N"], shape=[K, N], seed=seed + 1, density=[1, 1], name="B") # SpMM, B is dense

# Calculating A_NNZ
A_NNZ = 0

for row in A_MK.getRoot().getPayloads():
    for nz in row.getPayloads():
        A_NNZ += 1

print(f"A_NNZ: {A_NNZ}")

TOTAL_WORK = A_NNZ
WORK_PER_THREAD = math.ceil(TOTAL_WORK / (BLOCK_SIZE * GRID_SIZE))
print(f"TOTAL_WORK: {TOTAL_WORK}, WORK_PER_THREAD = {WORK_PER_THREAD}")

## TeAAL Specifications

Rows of matrix $A$ are partitioned across the SMs' warp/block. A thread warp/block can be assigned to a row with all zeros. 

Note that the current TeAAL specificaiton does not allow to specify the rank of `opt: slip`. This means there exists a synchronization across the SMs.

In [ ]:
yaml = """
einsum:
  declaration:
    A: [M, K]
    B: [K, N]
    Z: [M, N]
  expressions:
    - Z[m, n] = A[m, k] * B[k, n]
mapping:
  rank-order:
    A: [M, K]
    B: [K, N]
    Z: [M, N]
  partitioning:
    Z:
      (M, K): [flatten()]
      MK: [uniform_occupancy(A.WORK_PER_THREAD)]
  loop-order:
    Z: [MK1, MK0, N]
    # MK1: Number of iterations to process all NNZ in parallel
    # MK0: Number of NNZ assigned to each thread = WORK_PER_THREAD
  spacetime:
    Z:
      space: [MK1]
      time: [MK0, N]
"""

utils.compile(yaml)

## Check Results

Check that generated code computes the correct result.

**Note**: Should be used after compiling and running the kernel (above cell).

In [ ]:
utils.check_matmul(A_MK, B_KN, Z_MN)

## Performance on GPU

Load Balance: Better load balance than Thread-Mapped, since rows with high NNZ will be processed by a group of threads (smoothing out heavy rows). Additionally, there's no need to worry about load balance across the warps since SMs can simply start processing on another thread warp when one warp finishes earlier than the others.

Assuming that the $A$, $B$, and $Z$ are stored in CSR format, the memory access pattern would be:
- A: Coalesced access, threads in a warp are accessing the same row of $A$.
- B: Uncoalesced access, threads are accessing $B$ in column-wise order and at random positions based on column indices of $A$'s nonzero entries.
- Z: Coalesced access, threads in a warp are writing the same row of $Z$. At least with writing to $Z$, it may not be as fast as Thread-Mapped since it is done atomically on Group-Mapped to avoid data conflict. 